# Dirección Directiva — Analítica de Indicadores de Gestión PAI/MIPG

**Bloque A — Gestión institucional** | Línea: `direccion_directiva`

---

## Contexto institucional

La **gestión pública ambiental en Colombia** opera bajo el marco del **Sistema Nacional Ambiental (SINA)**, creado por la Ley 99/1993. Las Corporaciones Autónomas Regionales (CARs) y el MADS deben reportar anualmente su desempeño institucional mediante indicadores estandarizados.

Los dos marcos de medición más relevantes son:

| Marco | Entidad responsable | Qué mide |
|---|---|---|
| **PAI** (Plan de Acción Institucional) | Cada CAR / entidad | Avance físico y financiero de su plan cuatrienal |
| **MIPG** (Modelo Integrado de Planeación y Gestión) | DAFP (Función Pública) | Madurez en 7 dimensiones de gestión pública |
| **IEDI** (Índice de Evaluación del Desempeño Institucional) | MADS | Eficacia, eficiencia y capacidad de las CARs |
| **IMG** (Indicadores Mínimos de Gestión) | MADS / Res. 667/2016 | Relacionan la gestión con el estado de los recursos naturales |

**¿Por qué analizar estos indicadores estadísticamente?**  
Los informes PAI y MIPG son la base de la rendición de cuentas ambiental en Colombia. Detectar tendencias reales de mejora (o deterioro) en la gestión institucional, separándolas del ruido estadístico, permite priorizar inversiones, identificar CARs rezagadas y fundamentar decisiones de política con evidencia, no con reportes cualitativos.

---

## Preguntas analíticas que responde este notebook

1. ¿Ha mejorado sostenidamente el indicador compuesto de gestión en los últimos 10 años? (Mann-Kendall)
2. ¿Qué porcentaje de los 16 indicadores PAI está en zona roja (<60%), amarilla (60-79%) o verde (≥80%)?
3. ¿Existe correlación entre el desempeño MIPG y el estado de los recursos naturales gestionados?
4. ¿Cuáles son los indicadores con mayor variabilidad interanual — y por qué?

---

## Marco normativo relevante

- **Ley 99/1993:** Crea el SINA y el deber de reporte institucional.
- **Decreto 1076/2015:** Decreto Único Reglamentario del Sector Ambiente.
- **Resolución 667/2016 (MADS):** Establece los Indicadores Mínimos de Gestión para CARs.
- **Ley 1931/2018:** Gestión del cambio climático — crea RENARE y obliga a reportar GEI.

> **Documentación de dominio completa:** [`docs/fuentes/direccion_directiva.md`](../../docs/fuentes/direccion_directiva.md)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "direccion_directiva"
VARIABLE = "indicador"
UNIDAD = ""

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `direccion_directiva` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "direccion_directiva.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### Variable principal: indicadores de gestión PAI/MIPG (%)

**Estructura de datos recomendada para esta línea:**

```
fecha       | indicador              | valor_pct | semaforo
------------|------------------------|-----------|--------
2015-01-01  | avance_fisico_pai      | 72.3      | amarillo
2015-01-01  | ejecucion_financiera   | 85.1      | verde
2015-01-01  | restauracion_ha        | 45.2      | rojo
...         | ...                    | ...       | ...
```

**16 indicadores PAI típicos (Res. 667/2016):**
- Avance físico del plan cuatrienal (%)
- Ejecución financiera presupuestal (%)
- Áreas en proceso de restauración (ha)
- Hectáreas deforestadas en jurisdicción (ha)
- Concesiones de agua otorgadas vs. solicitadas
- Predios adquiridos para conservación (ha)
- Índice de riesgo de calidad del agua
- Proyectos de educación ambiental ejecutados

**Indicador compuesto recomendado:**  
Promedio ponderado de los 16 indicadores — el peso de cada uno puede seguir la ponderación oficial del IEDI (MADS) o una ponderación propia documentada.

**Frecuencia:** Anual (los informes PAI son cuatrienales con cortes anuales). Esta frecuencia tiene implicaciones metodológicas críticas — ver Sección 7.

**Fuentes de datos reales:**
- Informes PAI publicados en página web de cada CAR (sección "Rendición de cuentas")
- MADS: https://www.mads.gov.co → Gestión sectorial → Evaluación
- DAFP/FURAG: https://www.funcionpublica.gov.co → MIPG → Resultados

> Colocar el archivo en `data/raw/direccion_directiva.csv` y ajustar la ruta.

In [ ]:
# df = load_csv("data/raw/direccion_directiva.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "indicador": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

### ¿Qué validar en indicadores de gestión?

Los indicadores PAI/MIPG son porcentajes de cumplimiento (0–100%), pero tienen particularidades:

**Rangos válidos:**
- `valor_pct`: [0, 100] — ningún indicador puede superar el 100% de cumplimiento en reportes formales.
- Atención: algunos informes reportan "ejecución financiera > 100%" (traslados presupuestales). Documentar si se permite en la metodología de la entidad.

**Señales de alerta específicas de esta línea:**
- **Salto de 0 a 100 en un año:** improbable en gestión institucional — puede ser error de transcripción del informe.
- **Valor exactamente 0 o exactamente 100:** frecuente en indicadores binarios (ej. "¿se aprobó el plan?"). Documentar estos casos.
- **Indicadores ausentes en años de transición de gobierno:** los planes cuatrienales cambian cada 4 años — hay lagunas metodológicas en los años de empalme.
- **Cambio de metodología de cálculo:** el MADS actualiza las fórmulas del IEDI. Verificar que la serie sea comparable entre períodos.

> `validate()` aplica rangos [0,100] para esta línea (ADR-006).

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_direccion_directiva.html",
       title="EDA — Dirección Directiva", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria — Semáforo de cumplimiento

### Sistema de semáforo para indicadores PAI

La visualización central para esta línea no es una serie temporal predictiva, sino un **cuadro de mando de cumplimiento**:

| Color | Rango | Interpretación |
|---|---|---|
| Verde | ≥ 80% | Cumplimiento satisfactorio — meta alcanzada |
| Amarillo | 60% – 79% | Cumplimiento en riesgo — requiere intervención |
| Rojo | < 60% | Incumplimiento — requiere plan de mejora urgente |

Este semáforo es coherente con la metodología del IEDI (MADS) y es el estándar de reporte de las CARs ante la Contraloría.

**¿Qué patrones buscar en la visualización?**
- ¿Qué indicadores están consistentemente en rojo en los últimos 5 años? (problemas estructurales)
- ¿Hay indicadores que oscilan entre verde y rojo año a año? (inestabilidad de gestión)
- ¿El indicador compuesto muestra mejora al comparar inicio vs. final del período?
- ¿Hay correlación visual entre años electorales/cambio de gobierno y caídas en cumplimiento?

**Gráficos recomendados:**
- Heatmap de semáforos: indicadores (filas) × años (columnas), coloreado por rango.
- Línea del indicador compuesto anual con bandas de color.
- Barras apiladas: % de indicadores en verde/amarillo/rojo por año.

In [ ]:
plot_series(df, "fecha", "indicador", title="Dirección Directiva — indicador ()")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "indicador", period="month")
plt.show()

## 4. Estadística descriptiva — Análisis multivariado de indicadores

### Estadísticos relevantes para 16 indicadores × 10 años

Con una matriz de 16 × 10 = 160 observaciones por entidad, los estadísticos más útiles son:

**Por indicador (análisis de columnas):**
- Media anual y tendencia: ¿cuáles indicadores mejoran, cuáles empeoran?
- CV (coeficiente de variación): indicadores con CV >25% son inestables — preocupante en gestión institucional.
- P10 / P90: rangos de variación histórica — útiles para fijar metas realistas.

**Por año (análisis de filas):**
- Promedio compuesto: evolución del desempeño global.
- % en zona roja: ¿en qué años hubo más incumplimiento colectivo?
- Correlación entre indicadores: ¿los indicadores financieros y físicos coevolucionan?

**Estadísticos de advertencia:**
- **Indicadores con sesgo negativo extremo (skewness < -1):** la entidad tiene años excepcionales de alto cumplimiento pero la mayoría del tiempo está por debajo — patrón de "esfuerzo de fin de cuatrienio".
- **Moda en valores exactos (50%, 75%, 100%):** posible redondeo sistemático en los informes — sesgos de auto-reporte.

> La estadística descriptiva es el producto principal de esta línea, no los modelos predictivos.

In [ ]:
summarize(df)

## 5. Análisis inferencial — Mann-Kendall sobre el indicador compuesto

### ¿Hay mejora sostenida en la gestión institucional?

La pregunta central de esta sección es: **¿el indicador compuesto de gestión muestra una tendencia monótona estadísticamente significativa en los últimos 10 años?**

**Por qué Mann-Kendall (no regresión lineal simple):**
- Mann-Kendall es no paramétrico — no asume distribución normal de los residuales.
- Robusto ante outliers — un año atípico no distorsiona la conclusión de tendencia.
- Apropiado para series cortas (n ≥ 8) como series anuales de 10 años.
- La regresión lineal simple asume que los errores son i.i.d. — en indicadores institucionales los años no son independientes (hay correlación serial por continuidad de gestión).

**Interpretación del resultado:**
- `trend = "increasing"`, p < 0.05: hay evidencia estadística de mejora sostenida en gestión.
- `trend = "no trend"`, p > 0.05: no se puede afirmar mejora, aunque visualmente parezca crecer.
- `trend = "decreasing"`, p < 0.05: deterioro institucional sostenido — alerta crítica.
- `slope (Theil-Sen)`: magnitud de la mejora en puntos porcentuales por año.

**ADR-004 sobre estacionariedad:**
- Para indicadores de cumplimiento, el test ADF/KPSS tiene interpretación diferente que en series hidrológicas. Un indicador que mejora gradualmente será I(1), lo cual es esperado y deseable desde el punto de vista institucional.
- Si la serie es estacionaria (fluctúa sin tendencia), interpretarlo como estancamiento institucional.

> ADR-004: ADF + KPSS obligatorios antes de cualquier ARIMA. Para esta línea, Mann-Kendall es la prueba inferencial primaria.

In [ ]:
ts = df.set_index("fecha")["indicador"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Umbrales normativos de cumplimiento

### ¿Aplica `exceedance_report()` a indicadores de gestión?

Los indicadores PAI/MIPG no tienen umbrales en las normas de calidad ambiental (Res. 2254, Res. 2115). Sin embargo, **sí existen metas normativas de cumplimiento institucional:**

| Indicador | Umbral | Consecuencia del incumplimiento |
|---|---|---|
| Avance físico PAI | ≥ 80% al cierre del cuatrienio | Observaciones de la Contraloría |
| Ejecución financiera | ≥ 85% anual | Posibles recortes presupuestales |
| MIPG — dimensión "Gestión con valores" | ≥ 70 puntos | Alerta del DAFP |
| IEDI global (MADS) | ≥ 60 puntos | Plan de mejora obligatorio |

**Análisis de "no-excedencia de metas"** (lógica inversa a la de contaminación):
- En calidad del aire, exceder el umbral es malo.
- En indicadores de gestión, **no alcanzar** el umbral es el problema.
- `exceedance_report()` devuelve vacío (es esperado) — use umbrales manuales como se muestra a continuación.

```python
# Análisis manual de incumplimiento de metas
umbral_verde = 80
df["semaforo"] = pd.cut(df["indicador"],
                         bins=[-1, 60, 80, 101],
                         labels=["rojo", "amarillo", "verde"])
print(df["semaforo"].value_counts(normalize=True).mul(100).round(1))
```

In [ ]:
rep = exceedance_report(df["indicador"], variable="indicador")
if rep.empty:
    print("Sin normas colombianas registradas para 'indicador'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento — Manejo de la naturaleza de los datos de gestión

### Particularidades del preprocesamiento para indicadores institucionales

**ADR-002 (Outliers):** No hacer clipping. Un indicador en 0% puede ser real (meta no iniciada) o un error. Documentar caso a caso.

**Faltantes en reportes institucionales — causas frecuentes:**
1. **Transición de planes cuatrienales:** el primer año de un nuevo plan puede no tener línea base.
2. **Indicador nuevo:** se incorporó en la Res. 667/2016 pero la entidad no tenía datos históricos.
3. **Error administrativo:** el informe se publicó tarde o no está disponible digitalmente.
4. **Año de pandemia (2020):** muchos reportes PAI fueron suspendidos o prorrogados.

**Estrategia de imputación recomendada:**
- `method="linear"`: adecuado si el indicador tiene tendencia suave.
- **No usar** para indicadores binarios (0 o 100) — la imputación lineal generaría valores intermedios sin sentido.
- Para faltantes en años de transición de plan: imputar con el último valor conocido (`method="ffill"`).

**Normalización:**
- Los 16 indicadores tienen diferentes escalas (%, ha, número de proyectos). Para el indicador compuesto, normalizar a [0,100] antes de promediar o usar pesos oficiales del IEDI.

In [ ]:
df_clean = impute(df.copy(), cols=["indicador"], method="linear")
print(f"Faltantes antes: {df["indicador"].isna().sum()} | "
      f"después: {df_clean["indicador"].isna().sum()}")

## 7. Sobre los modelos predictivos — Por qué esta línea NO es predictiva

### Decisión metodológica fundamental

**Los indicadores PAI/MIPG NO son series temporales predictivas en el sentido convencional.**

Esta es una distinción crítica que diferencia esta línea de otras como calidad del aire u oferta hídrica:

| Característica | Series ambientales (ej. caudal) | Indicadores de gestión PAI |
|---|---|---|
| **Proceso generador** | Físico/natural, continuo | Administrativo, discreto |
| **Frecuencia** | Diaria / mensual | Anual |
| **Dependencia temporal** | Autocorrelación real (lluvia hoy depende de lluvia ayer) | Débil (el cumplimiento este año depende más de decisiones políticas que del año anterior) |
| **Predictibilidad** | Moderada-alta con modelos correctos | Baja — depende de factores exógenos (presupuesto, voluntad política) |
| **Objetivo del análisis** | Proyectar valores futuros | Diagnosticar el presente y detectar tendencias |

**Por qué RandomForest/XGBoost tienen limitaciones aquí:**
- Con solo 10 observaciones anuales, los modelos de ML no tienen datos suficientes para aprender patrones.
- Las covariables que realmente importan (presupuesto asignado, cambio de director, prioridades del gobierno) son difíciles de cuantificar.
- Un modelo que predice "el indicador compuesto será 74% el año que viene" tiene poco valor decisional.

**Cuándo SÍ es útil ARIMA aquí:**
- Para proyectar si se alcanzará una meta cuatrienal al ritmo actual (ej. "¿llegamos al 80% de restauración al 2027?").
- ARIMA(1,1,0) con d=1 (serie anual con tendencia) — sin componente estacional.
- Horizonte máximo confiable: 3–4 años (hasta el fin del plan cuatrienal).

**El valor real de la sección de modelos en esta línea:**
- Demostrar que el indicador compuesto tiene (o no) una tendencia estadísticamente extrapolable.
- Generar un intervalo de confianza para la meta de fin de cuatrienio.

> Registrar en `docs/decisiones.md` la decisión de no usar SARIMA ni ML complejo para esta línea.

In [ ]:
ts = df_clean.set_index("fecha")["indicador"]

models = {
    "RandomForest": get_model("random_forest", lags=[1,2,3,6,12]),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

### Plantilla de hallazgos (completar con datos reales)

| Aspecto | Hallazgo | Implicación |
|---|---|---|
| Semáforo general | ___% verde / ___% amarillo / ___% rojo | El ___% de los indicadores están en zona de riesgo |
| Tendencia compuesta | Mann-Kendall: ___ (p=___), slope=___pp/año | La gestión institucional ___ de forma estadísticamente ___ |
| Indicadores críticos | ___ (rojo en ___ de los últimos 10 años) | Requieren plan de mejora estructural |
| Indicadores exitosos | ___ (verde en ___ de los últimos 10 años) | Buenas prácticas replicables |
| Proyección cuatrienal | ARIMA → ___% en 2027 | ___ alcanzar meta del PAI actual |

### Decisiones metodológicas registradas

- **No se usó SARIMA:** indicadores de gestión son reportes anuales de cumplimiento, no series con ciclo estacional.
- **No se usó ML (RandomForest/XGBoost):** n=10 insuficiente para aprendizaje confiable sin covariables exógenas.
- **Mann-Kendall como prueba principal:** no paramétrico, robusto para series cortas y no normales.
- **Semáforo con umbrales institucionales:** verde≥80%, amarillo 60-79%, rojo<60% (coherente con IEDI/MADS).
- Registrar decisiones adicionales en `docs/decisiones.md`.

---

## 9. Cómo adaptar a datos reales

### Paso 1: Extraer indicadores de los informes PAI

Los informes PAI de las CARs se publican en PDF. La extracción puede hacerse de tres formas:

```python
# Opción A: PDF con tablas estructuradas
import pdfplumber

with pdfplumber.open("data/raw/informe_pai_2023.pdf") as pdf:
    for page in pdf.pages:
        tabla = page.extract_table()
        if tabla:
            df_raw = pd.DataFrame(tabla[1:], columns=tabla[0])

# Opción B: Descarga del FURAG (DAFP - MIPG)
# https://www.funcionpublica.gov.co/furag
# Exportar en Excel por entidad y dimensión

# Opción C: MADS - SISAIRE / VITAL
# Algunos indicadores están en bases de datos públicas
```

### Paso 2: Estructurar la matriz de indicadores

```python
# Estructura esperada: formato largo (tidy)
df = pd.DataFrame({
    "fecha": ["2014-01-01", "2015-01-01", ...],     # primer día del año
    "indicador": ["avance_fisico_pai", ...],
    "valor_pct": [72.3, ...],
    "meta_pct": [80.0, ...],
    "cumple": [False, ...]
})

# Calcular semáforo
df["semaforo"] = pd.cut(df["valor_pct"],
                         bins=[-1, 60, 80, 101],
                         labels=["rojo", "amarillo", "verde"])
```

### Paso 3: Calcular indicador compuesto

```python
# Ponderación igual (simplificado)
indicador_compuesto = df.groupby("fecha")["valor_pct"].mean().reset_index()
indicador_compuesto.columns = ["fecha", "indicador"]

# Ponderación por importancia (usar pesos del IEDI-MADS)
pesos = {"avance_fisico_pai": 0.30, "ejecucion_financiera": 0.25, ...}
df["valor_ponderado"] = df.apply(lambda r: r["valor_pct"] * pesos.get(r["indicador"], 1/16), axis=1)
```

### Fuentes de datos adicionales

- **FURAG (DAFP):** https://www.funcionpublica.gov.co → MIPG → Resultados por entidad.
- **Contraloría General:** informes de auditoría de CARs (contienen indicadores de ejecución).
- **MADS:** informes IEDI por CAR (publicados anualmente).
- **DANE-CSA:** Cuenta Satélite Ambiental — estadísticas macroeconómicas de gasto ambiental.